# E2 baseline: compare last-four-layer fusion with completed E3

Run **E2 only**, using the exact reconstructed inputs from the completed E3 run. E2 injects graphs into layers 8–11; E3 injects into all 12 layers. Both retain late graph concatenation.

Keep the same GPU class as E3 if available: **A100**. Seed 42, 10 epochs, batch 64, accumulation 8, learning rates, validation-F1 checkpoint selection, and threshold 0.5 remain the same. The runner checks input hashes, encoder revision, recorded dependency versions, and unchanged model/data/training code against E3 before training.

The hypothesis is that all-layer E3 improves F1 over E2 on functions with **0–64 model tokens, inclusive**. Fresh E3 already has short unseen-test F1 **0.400**, on 1,938 functions including 19 vulnerable ones. This run tests that comparison; it cannot reproduce the later E6 result of 0.744.

Use the existing `e3-inputs-20261014.zip` in `My Drive/data/graph-vul/e3-reproduction-20261014/`, and keep the completed E3 folder `runs/E3_all_layers-20261014T091812Z` there. No new input upload or E3 rerun is needed. Every execution creates a new E2 run folder and preserves existing artifacts.


In [ ]:
%pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -q numpy==2.2.6 pandas==2.3.3 scikit-learn==1.7.2 transformers==5.3.0 pytorch-lightning==2.6.1 torchmetrics==1.9.0 torch-geometric==2.7.0 sentencepiece==0.2.1 wandb==0.25.1 tqdm==4.67.3 matplotlib==3.10.8 seaborn==0.13.2


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, sys, zipfile

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/data/graph-vul/e3-reproduction-20261014")
BUNDLE = DRIVE_ROOT / "e3-inputs-20261014.zip"
REFERENCE = DRIVE_ROOT / "runs/E3_all_layers-20261014T091812Z"
for path in (BUNDLE, REFERENCE / "run.json", REFERENCE / "predictions/test_predictions.csv", REFERENCE / "predictions/unseen_test_predictions.csv"):
    if not path.is_file():
        raise FileNotFoundError(path)
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOCAL = Path("/content") / ("e2-inputs-" + STAMP)
LOCAL.mkdir(exist_ok=False)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(LOCAL)
PREPARED = LOCAL / "prepared"
print("Prepared inputs:", PREPARED)
print("Completed E3 reference:", REFERENCE)


In [ ]:
REPO = Path("/content") / ("graph-vul-e2-" + STAMP)
subprocess.run(["git", "clone", "--branch", "portfolio/appsflyer", "--single-branch", "https://github.com/noamdwc/graph-vul.git", str(REPO)], check=True)
RUNNER = REPO / "scripts/run_e3_reproduction.py"
print("Source revision:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before launching E2"
print(torch.cuda.get_device_name(0))
OUTPUT = DRIVE_ROOT / "runs" / ("E2_last4-" + STAMP)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
LOG = OUTPUT.parent / (OUTPUT.name + "-training.log")
print("Fresh output:", OUTPUT)
print("Full training log:", LOG)
env = os.environ.copy()
env["WANDB_MODE"] = "disabled"
command = [sys.executable, "-u", str(RUNNER), "--experiment", "E2", "--reference-run", str(REFERENCE / "run.json"), "--data-root", str(PREPARED), "--output-dir", str(OUTPUT)]
# Stream progress into the notebook and retain the complete log on Drive.
with LOG.open("x") as log, subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


In [ ]:
# Use the same fresh Python process strategy as training: the notebook kernel
# can retain NumPy modules imported before the pinned packages were installed.
subprocess.run([sys.executable, str(REPO / "scripts/compare_e2_e3_runs.py"),
                "--e2-run", str(OUTPUT), "--e3-run", str(REFERENCE)],
               cwd=REPO, check=True)
print("Also download the training log:", LOG)
print("Save this executed notebook with its outputs.")
